# Verifying Symbolic Pauli-Exponent Circuits with `PhasedOutcomeCompleteSimulation`

This notebook shows how to apply a **symbolic Pauli exponent** $e^{i\alpha P}$ to a
`PhasedOutcomeCompleteSimulation` and use it to verify equivalence of non-stabilizer circuits, the
motivating application of [arXiv:2603.24717](https://arxiv.org/abs/2603.24717). (See the companion
notebook *Tracking the Exact Global Phase* for an introduction to the phased simulator itself.)

## Applying a symbolic Pauli exponent

An arbitrary-angle Pauli exponent $e^{i\alpha P} = \cos(\alpha)\,I + i\sin(\alpha)\,P$ is **not** a
stabilizer operation, so it cannot be applied as an ordinary Clifford gate. (Note that
`apply_pauli_exp` is only the *fixed* Clifford exponent $e^{i\pi/4\,P}$, not a free-angle one.)

The simulator exposes it as a first-class operation parameterised by a **symbolic angle**: allocate the
angle with `allocate_symbolic_angle()`, then apply the exponent with `apply_symbolic_pauli_exp(P, angle)`.
The symbol $\alpha$ stands for the angle of *every* $e^{i\alpha P}$ at once. The simulator carries the
two halves of
$e^{i\alpha P}|\psi\rangle = \cos(\alpha)\,|\psi\rangle + i\sin(\alpha)\,P|\psi\rangle$
together, with their **exact** relative phase, for all $\alpha$ simultaneously. Two circuits agree for
every $\alpha$ exactly when these halves match, phase included.

Nothing here forms a $2^n$ state vector: phases are exact integer powers of $\zeta_8 = e^{i\pi/4}$ and
every step is polynomial in the number of qubits.

> A symbolic angle is an **opaque handle**, not a number. Each one carries an `index` -- its
> subscript $k$ in $\alpha_k$, fixed by allocation order -- and when two circuits are compared, angles
> sharing an `index` must correspond. Allocate a circuit's angles up front with
> `allocate_symbolic_angles(n)` and refer to them as `angles[k]`; the correspondence between two
> circuits is then explicit and independent of how either circuit is otherwise written.

## Setup

In [1]:
from paulimer import (
    PhasedOutcomeCompleteSimulation,
    PhasedCircuitAction,
    SparsePauli,
    UnitaryOpcode,
)

ZETA8_LABEL = {0: "1", 1: "ζ₈", 2: "i", 3: "ζ₈³", 4: "-1", 5: "ζ₈⁵", 6: "-i", 7: "ζ₈⁷"}

## A single symbolic Pauli exponent and its two halves

We build $C_1\, e^{i\alpha Z_0}\, C_2\,|0\rangle$ with $C_2 = H_0$ and $C_1 = H_0$. The simulator keeps
the identity half ($\cos\alpha$) and the $Z_0$ half ($i\sin\alpha$) of the exponent, each with its
exact $\zeta_8$ phase.

In [2]:
sim = PhasedOutcomeCompleteSimulation(1)
sim.apply_unitary(UnitaryOpcode.Hadamard, [0])              # C_2
alpha = sim.allocate_symbolic_angle()                       # the exponent's symbolic angle alpha
sim.apply_symbolic_pauli_exp(SparsePauli("Z_0"), alpha)     # e^{i alpha Z_0}
sim.apply_unitary(UnitaryOpcode.Hadamard, [0])              # C_1

for fired in (0, 1):
    exponent = sim.output_phase_exponent([bool(fired)])
    half = "identity half (weight cos a)" if fired == 0 else "Z half (weight i*sin a)"
    print(f"  {half}:  zeta8^{exponent} = {ZETA8_LABEL[exponent]}")

  identity half (weight cos a):  zeta8^0 = 1
  Z half (weight i*sin a):  zeta8^0 = 1


## Verifying equivalence over all inputs with Choi states

To check that two circuits implement the **same operator** on an *unknown* input, it is not enough to
run them on one fixed state: we must compare them on every input at once. Channel-state duality lets us
do this with a single stabilizer state. Entangle each of the $n$ system qubits with a fresh *reference*
qubit via a Bell pair $|\Phi\rangle$ (`prepare_bell_pairs` below), then apply the circuit to the system
qubits only. The resulting **Choi state** $(U \otimes I)\,|\Phi\rangle^{\otimes n}$ determines $U$
completely.

This is the same Bell-pair recipe used to verify Clifford circuits with `OutcomeCompleteSimulation`
(see *ZZ-measurement verification*); the only new ingredient is `apply_symbolic_pauli_exp`. Once the
circuit is applied, `sim.phased_action(input_qubits, output_qubits)` returns a `PhasedCircuitAction`,
and two actions are compared directly:

- `a.is_equivalent(b)`: equal as operators on every input, **including** the exact relative phases
  (up to one overall global phase);
- `a.is_equivalent_up_to_signs(b)`: equal only in their stabilizer (symplectic) action, ignoring all
  phases, i.e. precisely what an ordinary, phase-blind stabilizer simulation sees.

The comparison matches the symbolic angles by their **index**: $\alpha_k$ of one circuit must
correspond to $\alpha_k$ of the other.

In [3]:
def prepare_bell_pairs(sim, systems, references):
    """Entangle each system qubit with its reference qubit (the channel-state-duality setup)."""
    for system, reference in zip(systems, references):
        sim.apply_unitary(UnitaryOpcode.PrepareBell, [system, reference])


# H . e^{i alpha Z0} . H, as an operator on every input (system qubit 0, reference qubit 1).
sim = PhasedOutcomeCompleteSimulation(2)
prepare_bell_pairs(sim, [0], [1])
sim.apply_unitary(UnitaryOpcode.Hadamard, [0])
alpha = sim.allocate_symbolic_angle()
sim.apply_symbolic_pauli_exp(SparsePauli("Z_0"), alpha)
sim.apply_unitary(UnitaryOpcode.Hadamard, [0])
hzh = sim.phased_action([0], [0])

# e^{i alpha X0}, on the same Choi layout.
sim = PhasedOutcomeCompleteSimulation(2)
prepare_bell_pairs(sim, [0], [1])
alpha = sim.allocate_symbolic_angle()
sim.apply_symbolic_pauli_exp(SparsePauli("X_0"), alpha)
x_exp = sim.phased_action([0], [0])

assert hzh.is_equivalent(x_exp)
print("verified:  H . e^{i alpha Z} . H  ==  e^{i alpha X}   (as operators over all inputs)")

verified:  H . e^{i alpha Z} . H  ==  e^{i alpha X}   (as operators over all inputs)


## A multi-qubit, entangling equivalence

The idiom extends verbatim to **entangling** Pauli exponents of arbitrary weight -- no new machinery.
A clean example: conjugating a single-qubit exponent by a `CNOT` turns it into a two-qubit
$ZZ$ exponent,
$$\mathrm{CNOT}_{01}\; e^{i\alpha Z_1}\; \mathrm{CNOT}_{01} \;=\; e^{i\alpha Z_0 Z_1},$$
because $\mathrm{CNOT}_{01}$ conjugates $Z_1 \mapsto Z_0 Z_1$. We verify it as operators (Choi state,
$n=2$), and confirm that dropping the conjugation -- a bare $e^{i\alpha Z_1}$ -- is genuinely different.

In [4]:
# e^{i alpha Z0 Z1}                          (system qubits 0,1; reference qubits 2,3)
sim = PhasedOutcomeCompleteSimulation(4)
prepare_bell_pairs(sim, [0, 1], [2, 3])
alpha = sim.allocate_symbolic_angle()
sim.apply_symbolic_pauli_exp(SparsePauli("Z_0 Z_1"), alpha)
zz_direct = sim.phased_action([0, 1], [0, 1])

# CNOT01 . e^{i alpha Z1} . CNOT01
sim = PhasedOutcomeCompleteSimulation(4)
prepare_bell_pairs(sim, [0, 1], [2, 3])
sim.apply_unitary(UnitaryOpcode.ControlledX, [0, 1])
alpha = sim.allocate_symbolic_angle()
sim.apply_symbolic_pauli_exp(SparsePauli("Z_1"), alpha)
sim.apply_unitary(UnitaryOpcode.ControlledX, [0, 1])
zz_via_cnot = sim.phased_action([0, 1], [0, 1])

# e^{i alpha Z1} alone                        (conjugation dropped)
sim = PhasedOutcomeCompleteSimulation(4)
prepare_bell_pairs(sim, [0, 1], [2, 3])
alpha = sim.allocate_symbolic_angle()
sim.apply_symbolic_pauli_exp(SparsePauli("Z_1"), alpha)
z1_only = sim.phased_action([0, 1], [0, 1])

assert zz_direct.is_equivalent(zz_via_cnot)
print("verified:  e^{i alpha Z0Z1}  ==  CNOT01 . e^{i alpha Z1} . CNOT01   (entangling)")

assert not zz_direct.is_equivalent(z1_only)
print("verified:  e^{i alpha Z0Z1}  !=  e^{i alpha Z1}   (the CNOT conjugation genuinely matters)")

verified:  e^{i alpha Z0Z1}  ==  CNOT01 . e^{i alpha Z1} . CNOT01   (entangling)
verified:  e^{i alpha Z0Z1}  !=  e^{i alpha Z1}   (the CNOT conjugation genuinely matters)


## A phase difference that ordinary simulation misses

Finally, a difference that is *purely* a phase. The exponents $e^{+i\alpha Z}$ and $e^{-i\alpha Z}$
differ only by the sign of the Pauli, $+Z$ versus $-Z$. Since $+Z$ and $-Z$ have the **same symplectic
action**, an ordinary stabilizer simulation cannot tell them apart -- *even* under the Choi comparison
above. Yet they are physically different, differing by a relative $-1$ on the $Z$ half of the exponent.
The two comparisons make this explicit: `is_equivalent_up_to_signs` (phase-blind) reports them equal,
while `is_equivalent` (phase-aware) separates them.

In [5]:
# e^{+i alpha Z0}
sim = PhasedOutcomeCompleteSimulation(2)
prepare_bell_pairs(sim, [0], [1])
alpha = sim.allocate_symbolic_angle()
sim.apply_symbolic_pauli_exp(SparsePauli("Z_0"), alpha)
positive = sim.phased_action([0], [0])

# e^{-i alpha Z0}
sim = PhasedOutcomeCompleteSimulation(2)
prepare_bell_pairs(sim, [0], [1])
alpha = sim.allocate_symbolic_angle()
sim.apply_symbolic_pauli_exp(SparsePauli("-Z_0"), alpha)
negative = sim.phased_action([0], [0])

assert positive.is_equivalent_up_to_signs(negative)
print("Ignoring phase (is_equivalent_up_to_signs):  e^{+i alpha Z} and e^{-i alpha Z} INDISTINGUISHABLE")

assert not positive.is_equivalent(negative)
print("Tracking phase (is_equivalent):              e^{+i alpha Z} != e^{-i alpha Z}")

Ignoring phase (is_equivalent_up_to_signs):  e^{+i alpha Z} and e^{-i alpha Z} INDISTINGUISHABLE
Tracking phase (is_equivalent):              e^{+i alpha Z} != e^{-i alpha Z}


In [6]:
def exponent_half_phases(observable):
    """The exact zeta8 exponent on each half (identity, P) of e^{i alpha observable}."""
    sim = PhasedOutcomeCompleteSimulation(2)
    prepare_bell_pairs(sim, [0], [1])
    alpha = sim.allocate_symbolic_angle()
    sim.apply_symbolic_pauli_exp(observable, alpha)
    return tuple(sim.output_phase_exponent([bool(fired)]) for fired in (0, 1))


for name, observable in (("e^{+i alpha Z}", SparsePauli("Z_0")), ("e^{-i alpha Z}", SparsePauli("-Z_0"))):
    phases = exponent_half_phases(observable)
    labels = tuple(ZETA8_LABEL[power] for power in phases)
    print(f"{name} half phases: exponents {phases}  =  {labels}")
print("\nThe Z half differs by zeta8^4 = -1, exactly the relative phase ordinary simulation discards.")

e^{+i alpha Z} half phases: exponents (0, 0)  =  ('1', '1')
e^{-i alpha Z} half phases: exponents (0, 4)  =  ('1', '-1')

The Z half differs by zeta8^4 = -1, exactly the relative phase ordinary simulation discards.


## Executing a diagonal channel remotely: "ejection"

A channel that is **diagonal in the $Z$ basis** -- any number of $Z$-Pauli exponents together with
non-destructive $Z$-parity measurements -- can be executed **remotely**, the way a measurement-based or
lattice-surgery gadget would. Copy each system qubit onto a fresh ancilla with a `CNOT`, run the
diagonal channel on the **ancillas**, then **measure** each ancilla in the $X$ basis; a $-$ outcome
triggers a conditional $Z$ correction on the corresponding system qubit. Averaging over the measurement
outcomes, the channel the gadget implements on the inputs is exactly the direct channel.

The example below ejects a genuinely non-trivial three-qubit channel: three overlapping $Z$-Pauli
exponents,
$$e^{i\alpha Z_0}, \qquad e^{i\beta Z_1 Z_2}, \qquad e^{i\gamma Z_0 Z_1},$$
followed by a non-destructive measurement of the three-qubit parity $Z_0 Z_1 Z_2$. This mixes three
kinds of bit at once: the symbolic angles of the exponents (matched one-to-one against the direct
circuit), the non-destructive parity outcome (a measurement outcome, observed identically in both circuits), and
the destructive $X$ read-outs that drive the corrections (measurement outcomes, marginalized away). The phased
comparison certifies the ejected gadget equals the direct channel for all $\alpha, \beta, \gamma$.

In [7]:
def apply_z_diagonal_channel(sim, support, angles):
    """A Z-diagonal channel on three qubits: three overlapping Pauli exponents e^{i alpha_k P_k}
    parameterised by the given symbolic angles, then a non-destructive measurement of the three-qubit
    Z parity. The k-th exponent uses angles[k], so two circuits given angles with matching index apply
    the same channel."""
    q0, q1, q2 = support
    observables = (
        SparsePauli(f"Z_{q0}"),         # angles[0] . Z_q0
        SparsePauli(f"Z_{q1} Z_{q2}"),  # angles[1] . Z_q1 Z_q2
        SparsePauli(f"Z_{q0} Z_{q1}"),  # angles[2] . Z_q0 Z_q1
    )
    for angle, observable in zip(angles, observables):
        sim.apply_symbolic_pauli_exp(observable, angle)
    sim.measure(SparsePauli(f"Z_{q0} Z_{q1} Z_{q2}"))   # non-destructive 3-qubit parity measurement


system = [0, 1, 2]           # system qubits (their references are 3, 4, 5)
ancilla = [6, 7, 8]

# Direct: run the diagonal channel straight on the system qubits.
direct = PhasedOutcomeCompleteSimulation(6)
prepare_bell_pairs(direct, system, [3, 4, 5])
angles = direct.allocate_symbolic_angles(3)             # alpha_0, alpha_1, alpha_2
apply_z_diagonal_channel(direct, system, angles)
direct_action = direct.phased_action(system, system)

# Ejected: copy each system qubit onto an ancilla, run the channel on the ancillas, then read the
# ancillas out in X and correct. Allocating the angles the same way makes alpha_k here the same alpha_k
# as in the direct circuit -- the comparison pairs them by index.
ejected = PhasedOutcomeCompleteSimulation(9)
prepare_bell_pairs(ejected, system, [3, 4, 5])
angles = ejected.allocate_symbolic_angles(3)            # the same alpha_0, alpha_1, alpha_2
for system_qubit, ancilla_qubit in zip(system, ancilla):
    ejected.apply_unitary(UnitaryOpcode.ControlledX, [system_qubit, ancilla_qubit])
apply_z_diagonal_channel(ejected, ancilla, angles)
for system_qubit, ancilla_qubit in zip(system, ancilla):
    outcome = ejected.measure(SparsePauli(f"X_{ancilla_qubit}"))
    ejected.apply_conditional_pauli(SparsePauli(f"Z_{system_qubit}"), [outcome])
ejected_action = ejected.phased_action(system, system)

assert direct_action.is_equivalent(ejected_action)
print("verified:  three-qubit Z-diagonal channel (3 Pauli exponents + a 3-qubit parity measurement)")
print("           ejected through ancillas == the channel applied directly")

verified:  three-qubit Z-diagonal channel (3 Pauli exponents + a 3-qubit parity measurement)
           ejected through ancillas == the channel applied directly


## Summary

- A symbolic Pauli exponent $e^{i\alpha P}$ is added by allocating an opaque angle handle with
  `allocate_symbolic_angle()` (or a batch with `allocate_symbolic_angles(n)`) and applying
  `apply_symbolic_pauli_exp(P, angle)`. This works for an **arbitrary** Pauli $P$ of any weight:
  multi-qubit and entangling exponents need nothing new, and a single angle may parameterise several
  exponents to model a shared $\alpha$. Two circuits' angles correspond when they share an `index`
  ($\alpha_k$), so allocating each circuit's angles up front and indexing them keeps the comparison
  explicit.
- To compare two circuits as **operators** on an unknown input, build their **Choi states**
  (`prepare_bell_pairs` to entangle every system qubit with a reference, then apply the circuit to the
  system qubits) and call `sim.phased_action(...)`. This is the same Bell-pair recipe used to verify
  Clifford circuits with `OutcomeCompleteSimulation`. The resulting `PhasedCircuitAction` objects
  compare with `is_equivalent` (phase-aware) and `is_equivalent_up_to_signs` (phase-blind).
- The phased simulator tracks both halves of every exponent with their **exact** $\zeta_8$ phase,
  precisely the information ordinary stabilizer simulation throws away. Here it is the only thing
  distinguishing $e^{+i\alpha Z}$ from $e^{-i\alpha Z}$, whose Paulis $+Z$ and $-Z$ share a symplectic
  action.
- A whole $Z$-diagonal channel -- several Pauli exponents plus non-destructive parity measurements --
  can be **ejected** onto ancillas and realised through $X$ read-outs and Pauli corrections; the phased
  comparison certifies the resulting channel equals the direct one for every choice of angles.